# Convergence diagnostics

Split-R̂, bulk and tail ESS, and MCSE following Vehtari et al. (2021), in numpy. `diagnose`
produces a `ConvergenceReport` with the thresholds it used; non-finite draws fail the verdict
outright. `to_inference_data` hands a posterior to ArviZ when it is installed.

In [ ]:
import numpy as np

from axiom.core import is_failure
from axiom.infer import (
    ConvergenceReport, ConvergenceThresholds, ParameterDiagnostics, Posterior, diagnose, ess_bulk, ess_tail,
    from_inference_data, mcse_mean, split_rhat, to_inference_data,
)

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

In [ ]:
rng = np.random.default_rng(0)
good = rng.normal(size=(4, 1000))
bad = good + np.array([0.0, 0.0, 1.5, 1.5])[:, None]
ar = np.zeros((4, 1000))
for c in range(4):
    for t in range(1, 1000):
        ar[c, t] = 0.9 * ar[c, t - 1] + rng.normal() * np.sqrt(1 - 0.81)
table(
    [
        [name, f"{split_rhat(d):.3f}", f"{ess_bulk(d):.1f}", f"{ess_tail(d):.1f}", f"{mcse_mean(d):.4f}"]
        for name, d in (("iid", good), ("shifted", bad), ("AR(0.9)", ar))
    ],
    headers=("draws", "split R-hat", "ESS bulk", "ESS tail", "MCSE"),
)

In [ ]:
post = Posterior({"mu": good, "tau": np.exp(ar)}, provenance={"seed": 0, "divergences": 0})
report: ConvergenceReport = diagnose(post)
print(report.converged, report.thresholds)
print(report.to_frame())
row: ParameterDiagnostics = report.row("tau")
print(row.passes(ConvergenceThresholds(rhat_max=1.05, ess_min=100.0)))
loose = diagnose(post, ess_min=100.0)
print("with ess_min=100:", loose.converged, loose.failing)

In [ ]:
idata = to_inference_data(post)
if not is_failure(idata):
    back = from_inference_data(idata)
    print(type(idata).__name__, back.names(), back == post)
else:
    print(idata)

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
import numpy as np

from axiom.core import Posterior
from axiom.display import show
from axiom.infer import diagnose
from axiom.viz import convergence

rng = np.random.default_rng(0)
posterior = Posterior({"alpha": rng.normal(size=(4, 600)), "beta": rng.normal(size=(4, 600))})
report = diagnose(posterior)
show(report)
convergence(report)

The question a sampler's output poses is *may I use this*, and the answer is per parameter: one bad row is enough.